# 2. 8비트 양자화된 .tflite 모델로 변환

- 8비트 양자화이지만, 입출력은 여전히 Float32 이다.
  - 문제는, ESP32 같은 장비에선 입출력이 실수인데 연산은 정수인 동적 양자화 모델이 작동되지 않는다는 것이다.
  - 실제 작동을 원한다면, TensorFlow 로 모델을 직접 구현하고 학습한 후, 입출력까지 완전정수양자화 (integer-only quantization) 할 것
  - PyTorch 모델의 경우 해당 구현을 지원하지 않으므로, 동적 양자화만 우선 다룬다.

- 이 부분부터는 Colab 에서 동작하지 않음
- 반드시 CPU torch 환경에서 진행할 것

In [ ]:
!pip install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cpu

In [ ]:
!pip install ipywidgets tqdm==4.67.1 ai-edge-torch==0.6.0

In [18]:
from transformers import AutoModelForImageClassification
from peft import PeftModel
import torch.nn as nn
import torch
import ai_edge_torch
from ai_edge_torch.generative.quantize import quant_recipes
from datetime import datetime
import os
import numpy as np
import random
import torchvision.transforms as transforms
import torchvision
import time
from ai_edge_litert.interpreter import Interpreter

MODEL_STATE_DICT_DIR_PATH = "./models_state_dict"

# cpu 로 가져오기
model_name = "facebook/deit-tiny-patch16-224"
loaded_model = AutoModelForImageClassification.from_pretrained(
    model_name,
    device_map="cpu",
)
loaded_model.classifier = nn.Linear(in_features=loaded_model.classifier.in_features, out_features=10)
# 모든 state 를 cpu 로 로드하기
loaded_model = PeftModel.from_pretrained(
    loaded_model,
    MODEL_STATE_DICT_DIR_PATH,
    device_map="cpu",
)
# LoRA 병합
loaded_model = loaded_model.merge_and_unload()
# cpu 로 모델을 보내야 함
loaded_model.to("cpu")
# 평가모드로 변경 필수
loaded_model.eval()

# 모든 연산자는 cpu 에서 실행되어야 함
# 배치사이즈는 1 필수
sample_args = (torch.randn(1, 3, 224, 224).to("cpu"),)

# 8비트 동적 양자화
quant_config = quant_recipes.full_int8_dynamic_recipe()
edge_model = ai_edge_torch.convert(
    loaded_model, sample_args, quant_config=quant_config
)

INFO:tensorflow:Assets written to: /tmp/tmpurbflydo/assets


INFO:tensorflow:Assets written to: /tmp/tmpurbflydo/assets
W0000 00:00:1761803666.101240   48606 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1761803666.101286   48606 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2025-10-30 14:54:26.101417: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpurbflydo
2025-10-30 14:54:26.103354: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2025-10-30 14:54:26.103363: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpurbflydo
2025-10-30 14:54:26.119436: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2025-10-30 14:54:26.244770: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpurbflydo
2025-10-30 14:54:26.272641: I tensorflow/cc/saved_model/loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 171228

In [19]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# 모델 가중치를 저장할 디렉터리 및 파일
CONVERTED_MODELS_DIR_PATH = "./converted_models"
BASE_FILE_NAME = "facebook_deit-tiny-patch16-224_cifar10_finetuned.tflite"
FILE_NAME = f"{timestamp}_{BASE_FILE_NAME}"

os.makedirs(CONVERTED_MODELS_DIR_PATH, exist_ok=True)

FULL_PATH = os.path.join(CONVERTED_MODELS_DIR_PATH, FILE_NAME)

edge_model.export(FULL_PATH)

# 3. 최종 테스트

In [23]:
# 전체 테스트할 이미지 갯수
MAX_TEST_SAMPLES = 5

# 시드 고정
torch.manual_seed(42)
torch.cuda.manual_seed(42)
np.random.seed(42)
random.seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# CIFAR-10 의 평균과 표준편차
mean = torch.tensor([0.4914, 0.4822, 0.4465])
std = torch.tensor([0.2470, 0.2435, 0.2616])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

testset  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=test_transform)

print(f"testset 이미지 갯수: {len(testset)}")
print(f"실제 테스트할 이미지 갯수: {MAX_TEST_SAMPLES}")

# 타임스탬프 방식이므로, 직접 수정 필요
TFLITE_MODEL_PATH = './converted_models/20251030_145432_facebook_deit-tiny-patch16-224_cifar10_finetuned.tflite'

# .tflite 작동을 위해선 배치사이즈 1 필수
testloader = torch.utils.data.DataLoader(testset, batch_size=1, shuffle=False)

interpreter = Interpreter(model_path=TFLITE_MODEL_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

correct = 0
total = 0
total_inference_time = 0.0

idx = 0

print("================================")
print("테스트 실행")
print("================================")

for inputs_torch, labels_torch in testloader:

    # MAX_TEST_SAMPLES 크기만큼만 테스트합니다.
    if idx >= MAX_TEST_SAMPLES:
        break

    input_data_fp32 = inputs_torch.numpy().astype(np.float32)

    # 입력 텐서 설정 및 추론 실행
    interpreter.set_tensor(input_details['index'], input_data_fp32)

    # 하나의 이미지의 실행시간 계산
    start_time = time.time()
    interpreter.invoke()
    end_time = time.time()

    total_inference_time += (end_time - start_time)

    print(f"img_idx: {idx}")
    print(f"time: {end_time - start_time}")
    print("================================")
    idx += 1

    # 출력 텐서 가져오기
    output_data_fp32 = interpreter.get_tensor(output_details['index'])
    
    # 예측 결과 및 정확도 계산
    predicted = np.argmax(output_data_fp32, axis=1) # 가장 높은 값의 인덱스 (클래스)
    labels_np = labels_torch.numpy()
    
    total += labels_np.shape[0]
    correct += (predicted == labels_np).sum()

print("테스트 종료")
print("================================")

# 최종 정확도 계산 및 출력
tflite_acc = 100 * correct / total
print(f"\n--- TFLite 모델 평가 결과 ---")
print(f"TFLite 모델 테스트 정확도: {tflite_acc:.2f}%")

if total > 0:
    avg_inference_time = total_inference_time / total
    print(f"평균 단일 이미지 추론 시간: {avg_inference_time:.4f}초")

testset 이미지 갯수: 10000
실제 테스트할 이미지 갯수: 5
테스트 실행
img_idx: 0
time: 0.017068147659301758
img_idx: 1
time: 0.014775276184082031
img_idx: 2
time: 0.013025283813476562
img_idx: 3
time: 0.011052608489990234
img_idx: 4
time: 0.014561176300048828
테스트 종료

--- TFLite 모델 평가 결과 ---
TFLite 모델 테스트 정확도: 100.00%
평균 단일 이미지 추론 시간: 0.0141초


- 전체 테스트를 하고싶다면 `MAX_TEST_SAMPLES` 를 10,000 까지 수정해볼 수 있다.
- 임베디드 환경을 가정하고 싶다면, VirtualBox 같은 걸 사용해, CPU 1 Core, Mem 1GiB 이하에서 동작시켜보자.